# M16 — Stanford Cars / RanPAC Phase-2 / ResNet-50 (locked Colab run)

Set **Runtime type = GPU** and run from top to bottom. The notebook uses `/content` only (no Google Drive), locks the official RanPAC source and ResNet-50 checkpoint, selects Ridge from the first training task, authorizes the protocol, and only then extracts test features. The public Kaggle mirror contains both raw splits, but no test image is enumerated or passed through the backbone before authorization. Three analytic batches create downloadable handoffs after 6 and 12 completed units. M16 is a Phase-2 backend comparison, not a full PETL reproduction.

In [ ]:
REPO_GIT_URL='https://github.com/ZaPhat206/SOHO-CL.git'
REPO_COMMIT='209f8f901fcf248530d25d0ca36f4f7bd87f05fd'
WORK_DIR='/content/SOHO-CL'
CACHE_DIR='/content/srq_m16_cars_features'
OUTPUT_DIR='/content/srq_m16_output'
SELECTION='/content/srq_m16_selection.json'
AUTHORIZATION='/content/srq_m16_authorization.json'
EXPORT='/content/srq_generalization_m16_cars_phase2_locked.zip'
HANDOFF='/content/m16_handoff_checkpoint.zip'
CONFIG='configs/srq_generalization_m16_cars_phase2_locked.json'
RUNNER='tools/srq_generalization_m16.py'
CHECKPOINT='/content/resnet50-11ad3fa6.pth'
CHECKPOINT_URL='https://download.pytorch.org/models/resnet50-11ad3fa6.pth'
CHECKPOINT_SIZE=102540417
CHECKPOINT_SHA='11ad3fa62ca79e40addfd354a8ec4b7c75143b3038b8d2a807fbc68deab379ca'
EXPECTED_SOURCE={'configs/srq_generalization_m16_cars_phase2_locked.json':'05d7e81df757fbd108faac7841fa62435e9c25c5b8e1f18c029a876acda8bba9','tools/srq_generalization_m16.py':'40ec521f23001451617cd1a7f9795b552be7b3a97decba7ba9a641d4e52f0a20'}
UPSTREAM_COMMIT='cf4b301d18b0c27db030f4371b72b768005ae58a'
UPSTREAM_SHA={'RanPAC.py':'36ba66a6ce6993b5830539a511e84dbd2b4994d8ebe2a0d5024ec5438a427e11','resnet.py':'618baf2cbd265fefbe549f244b53aa47f5cefa2bf76a9648e439703e715e9ed7','utils/data.py':'a181af7fae796a0d6c8cb59d8b6950d5bd0e54c5269eb158d02282bb4a5cdbca','args/cars_publish.csv':'a24fac27639ad3f2777c6324dd75a0e0ba163b00722bc014cc06ec6b4fa38047'}
IMPORT_HANDOFF=False  # Set True only to resume from a previously downloaded M16 handoff.
BATCH_SIZE=128
NUM_WORKERS=2

In [ ]:
import hashlib,json,os,shutil,subprocess,sys,urllib.request,zipfile
from pathlib import Path
def sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()
if Path(WORK_DIR).exists(): shutil.rmtree(WORK_DIR)
subprocess.run(['git','clone','--quiet',REPO_GIT_URL,WORK_DIR],check=True)
subprocess.run(['git','checkout','--quiet',REPO_COMMIT],cwd=WORK_DIR,check=True)
assert subprocess.check_output(['git','rev-parse','HEAD'],cwd=WORK_DIR,text=True).strip()==REPO_COMMIT
for relative,expected in EXPECTED_SOURCE.items(): assert sha(Path(WORK_DIR)/relative)==expected,(relative,sha(Path(WORK_DIR)/relative),expected)
assert not subprocess.check_output(['git','status','--porcelain'],cwd=WORK_DIR,text=True).strip()
os.chdir(WORK_DIR)
subprocess.run([sys.executable,'-m','pip','install','-q','kagglehub','timm','safetensors'],check=True)
print('M16 PINNED SOURCE: PASS',REPO_COMMIT)

In [ ]:
# Verify the exact official RanPAC files that define the claimed Phase-2 path.
UPSTREAM_DIR='/content/RanPAC-upstream'
if Path(UPSTREAM_DIR).exists(): shutil.rmtree(UPSTREAM_DIR)
subprocess.run(['git','clone','--quiet','https://github.com/McDonnell-Research-Lab/RanPAC.git',UPSTREAM_DIR],check=True)
subprocess.run(['git','checkout','--quiet',UPSTREAM_COMMIT],cwd=UPSTREAM_DIR,check=True)
for relative,expected in UPSTREAM_SHA.items(): assert sha(Path(UPSTREAM_DIR)/relative)==expected,(relative,sha(Path(UPSTREAM_DIR)/relative),expected)
cars_csv=(Path(UPSTREAM_DIR)/'args/cars_publish.csv').read_text().splitlines()
assert cars_csv[11].startswith('10,cars,True,16,20,ncm,pretrained_resnet50') and ',True,10000,True' in cars_csv[11]
print('M16 OFFICIAL RANPAC SOURCE LOCK: PASS')

In [ ]:
# Download the public mirror. This makes raw test files physically present, but the runner does not enumerate/read them before authorization.
import kagglehub
DATA_ROOT=kagglehub.dataset_download('eduardo4jesus/stanford-cars-dataset')
print('DATA ROOT:',DATA_ROOT)
if not Path(CHECKPOINT).is_file(): urllib.request.urlretrieve(CHECKPOINT_URL,CHECKPOINT)
assert Path(CHECKPOINT).stat().st_size==CHECKPOINT_SIZE
assert sha(CHECKPOINT)==CHECKPOINT_SHA
print('M16 RESNET-50 CHECKPOINT: PASS')

In [ ]:
# Extract only official training-split features.
subprocess.run([sys.executable,'-B',RUNNER,'extract-train','--config',CONFIG,'--feature-cache-dir',CACHE_DIR,'--root',DATA_ROOT,'--backbone-checkpoint',CHECKPOINT,'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS),'--require-clean-git'],check=True)
assert Path(CACHE_DIR,'train.pt').is_file() and not Path(CACHE_DIR,'test.pt').exists()
print('M16 TRAIN CACHE ONLY: PASS')

In [ ]:
# Select one first-task Ridge per paired replicate using train only, then freeze it.
subprocess.run([sys.executable,'-B',RUNNER,'select-ridge','--config',CONFIG,'--feature-cache-dir',CACHE_DIR,'--selection',SELECTION,'--device','cuda','--require-clean-git'],check=True)
selection=json.loads(Path(SELECTION).read_text())
assert selection['status']=='LOCKED_M16_TRAIN_ONLY_RIDGE_SELECTION' and selection['identity']['uses_test_set'] is False
assert not Path(CACHE_DIR,'test.pt').exists()
print('M16 RIDGE SELECTION: PASS',[(r['replicate']['class_order_seed'],r['selected_ridge_lambda']) for r in selection['identity']['records']])

In [ ]:
# Create immutable authorization before any test feature exists.
subprocess.run([sys.executable,'-B',RUNNER,'authorize','--config',CONFIG,'--feature-cache-dir',CACHE_DIR,'--selection',SELECTION,'--authorization',AUTHORIZATION,'--require-clean-git'],check=True)
authorization=json.loads(Path(AUTHORIZATION).read_text())
assert authorization['authorized'] is True and authorization['uses_test_set'] is False
assert not Path(CACHE_DIR,'test.pt').exists()
print('M16 AUTHORIZATION: PASS',authorization['authorization_id'])

In [ ]:
# Only this cell is authorized to enumerate and encode the official test split.
subprocess.run([sys.executable,'-B',RUNNER,'extract-test','--config',CONFIG,'--feature-cache-dir',CACHE_DIR,'--selection',SELECTION,'--authorization',AUTHORIZATION,'--root',DATA_ROOT,'--backbone-checkpoint',CHECKPOINT,'--device','cuda','--batch-size',str(BATCH_SIZE),'--num-workers',str(NUM_WORKERS),'--require-clean-git'],check=True)
assert Path(CACHE_DIR,'test.pt').is_file()
print('M16 AUTHORIZED TEST CACHE: PASS')

In [ ]:
# Optional cross-session resume: upload a previously downloaded handoff after reproducing train/test caches and authorization.
if IMPORT_HANDOFF:
    from google.colab import files
    uploaded=files.upload()
    assert len(uploaded)==1
    handoff_path=Path('/content')/next(iter(uploaded))
    handoff_path.write_bytes(next(iter(uploaded.values())))
    with zipfile.ZipFile(handoff_path) as archive:
        names=archive.namelist()
        assert 'handoff_manifest.json' in names and all(name=='handoff_manifest.json' or (name.startswith('units/') and name.endswith('.json') and '..' not in name) for name in names)
        manifest=json.loads(archive.read('handoff_manifest.json'))
        assert manifest['study_id']=='srq-generalization-m16-cars-ranpac-phase2-locked-v1'
        assert manifest['source_commit']==REPO_COMMIT and manifest['config_sha256']==EXPECTED_SOURCE[CONFIG]
        unit_dir=Path(OUTPUT_DIR)/'units'; unit_dir.mkdir(parents=True,exist_ok=True)
        for item in manifest['units']:
            payload=archive.read(item['name']); assert hashlib.sha256(payload).hexdigest()==item['sha256']
            (unit_dir/Path(item['name']).name).write_bytes(payload)
    print('IMPORTED M16 UNITS:',len(list((Path(OUTPUT_DIR)/'units').glob('*.json'))))

In [ ]:
# Run/resume in batches of six new units. Save the 6- and 12-unit handoffs locally.
from google.colab import files
for batch in range(3):
    command=[sys.executable,'-B',RUNNER,'run','--config',CONFIG,'--feature-cache-dir',CACHE_DIR,'--selection',SELECTION,'--authorization',AUTHORIZATION,'--output-dir',OUTPUT_DIR,'--export',EXPORT,'--handoff-export',HANDOFF,'--max-new-units','6','--device','cuda','--require-clean-git']
    completed=subprocess.run(command)
    assert completed.returncode==0,'M16 runner failed; preserve complete output.'
    count=len(list((Path(OUTPUT_DIR)/'units').glob('*.json')))
    print('M16 COMPLETED UNITS:',count,'/ 18')
    if Path(EXPORT).is_file(): break
    assert Path(HANDOFF).is_file() and count in (6,12)
    snapshot=Path('/content')/f'm16_handoff_{count:03d}_units.zip'
    shutil.copyfile(HANDOFF,snapshot)
    print('DOWNLOAD HANDOFF:',snapshot,'SHA256:',sha(snapshot))
    files.download(str(snapshot))
assert Path(EXPORT).is_file(),'M16 did not complete all 18 units.'

In [ ]:
result=json.loads(Path(OUTPUT_DIR,'m16_results.json').read_text())
print('STATUS:',result['status'])
print('RIDGES:',json.dumps(result['selected_ridge_by_class_order_seed'],indent=2))
print('SUMMARY:',json.dumps(result['summary'],indent=2))
print('GATES:',json.dumps(result['gates'],indent=2))
assert result['status']=='PASS_M16_CARS_RANPAC_PHASE2_LOCKED'
print('FINAL EXPORT:',EXPORT,'SHA256:',sha(EXPORT),'SIZE:',Path(EXPORT).stat().st_size)
files.download(EXPORT)